# Whisper Medium LoRA Levantine Custom Run: Palestine + Jordan + North Levantine

This notebook mirrors the practical structure of the original Whisper Medium mini run, but swaps in the exact Levantine datasets requested for training and held-out evaluation.

Custom dataset plan:
- train on the requested Casablanca Palestine/Jordan validation parquet files
- train on the requested Omnilingual APC North Levantine Arrow files (`data-00000`, `data-00001`)
- split the requested Casablanca test parquet files 50/50 into validation and test
- split Omnilingual `data-00002-of-00003.arrow` 50/50 into validation and test
- train for up to 50 epochs with early stopping patience `3` on rising validation loss


## 1. Config

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path("/home/MohammadNabulsi/whisper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    RUN_DIR,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = False
TRAIN_HOURS_CAP = 50.0
EVAL_SAMPLE_CAP = 100
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    train_hours_cap=TRAIN_HOURS_CAP,
    eval_sample_cap=EVAL_SAMPLE_CAP,
    num_train_epochs=50,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)
ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print("Notebook path:", SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print("Log path:", log_path)


{
  "model_name": "openai/whisper-medium",
  "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
  "sample_rate": 16000,
  "min_audio_seconds": 0.3,
  "drop_audio_at_or_above_seconds": 30.0,
  "train_hours_cap": 50.0,
  "eval_sample_cap": 100,
  "train_batch_size": 4,
  "eval_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "num_train_epochs": 50,
  "logging_steps": 5,
  "checkpoint_every_minutes": 5,
  "save_total_limit": 6,
  "generation_max_new_tokens": 256,
  "early_stopping_patience": 3,
  "split_seed": 42,
  "train_dataloader_num_workers": 4,
  "language": "ar",
  "task": "transcribe",
  "smoke_mode": false,
  "run_baseline_before_train": true,
  "run_post_train_eval": true,
  "force_rebuild_manifests": false,
  "resume_from_checkpoint": false,
  "use_fp16": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "q_proj",
  

## 2. Build Filtered Mini Manifests

In [2]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import prepare_manifests

manifest_state = prepare_manifests(config)
selection_summary = manifest_state["selection_summary"]
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


2026-06-23 14:31:53,810 | INFO | whisper_medium_run | prepared split=train rows=1860 kept=1541 dropped_empty=0 dropped_duration=319
2026-06-23 14:31:53,848 | INFO | whisper_medium_run | prepared split=eval_pool rows=1515 kept=1514 dropped_empty=0 dropped_duration=1
2026-06-23 14:31:53,889 | INFO | whisper_medium_run | prepared split=eval_pool rows=172 kept=19 dropped_empty=0 dropped_duration=153


{
  "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
  "drop_segments_at_or_above_seconds": 30.0,
  "full_counts_after_filter": {
    "train": 1541,
    "eval_parquet_pool": 1514,
    "eval_arrow_pool": 19
  },
  "selected_counts": {
    "train": 1541,
    "val": 766,
    "test": 767
  },
  "selected_hours": {
    "train": 2.1407003992305276,
    "val": 1.0070916678090185,
    "test": 1.037511566857676
  },
  "train_by_source_group": {
    "casablanca_levantine_train": {
      "rows": 1514,
      "hours": 2.0004557985360836
    },
    "omnilingual_apc_north_levantine_train": {
      "rows": 27,
      "hours": 0.14024460069444444
    }
  },
  "val_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "hours": 0.9691543645682779
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 9,
      "hours": 0.037937303240740745
    }
  },
  "test_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "h

## 3. Baseline Whisper Medium Predictions

In [3]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import load_whisper_bundle, run_predictions

baseline_bundle = load_whisper_bundle(config)
baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state["test_rows"],
        config,
        baseline_bundle,
        name="baseline_test_predictions",
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


/home/MohammadNabulsi/whisper/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


{
  "num_predictions": 767,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/eval_predictions/baseline_test_predictions.jsonl",
  "wer": 0.6661614567561351,
  "cer": 0.3663500651850712,
  "wer_loose": 0.6460718246316863,
  "cer_loose": 0.3587896563840253,
  "wer_no_punct": 0.6364614773961794,
  "cer_no_punct": 0.34677424366427984,
  "total_hours": 1.037511566857676,
  "by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "wer": 0.6611673466900462,
      "cer": 0.36130269693144934
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 10,
      "wer": 1.044215588759067,
      "cer": 0.7484358419842423
    }
  },
  "object_dump_predictions": 0,
  "object_dump_prediction_uids": []
}


## 4. Train LoRA Adapters

In [4]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import train_model

training_summary = train_model(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/pipeline.py:1056: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast toke

Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.764800,1.740716,0.600609,0.293178
2,1.009300,0.908952,0.512972,0.231348
3,0.541000,0.595583,0.486419,0.218510
4,0.482800,0.578262,0.482191,0.223080
5,0.407100,0.574458,0.501274,0.220375
6,0.336300,0.586980,0.472878,0.209550
7,0.311100,0.595434,0.522810,0.231658
8,0.298800,0.606513,0.483881,0.217769


/home/MohammadNabulsi/whisper/.venv/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'suppress_tokens': []}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call 

{
  "backend": "transformers",
  "best_checkpoint": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/checkpoints/checkpoint-485",
  "best_metric": 0.574457585811615,
  "completed_at": "2026-06-23T18:11:12Z",
  "train_rows": 1541,
  "val_rows": 766
}


## 5. Tuned Validation and Test Predictions

In [6]:
from pathlib import Path
import json

from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import (
    BEST_MODEL_DIR,
    load_whisper_bundle,
    run_predictions,
)

adapter_path = Path(training_summary["best_checkpoint"])
if not (adapter_path / "adapter_config.json").exists():
    adapter_path = BEST_MODEL_DIR

tuned_bundle = load_whisper_bundle(config, adapter_path=adapter_path)

val_prediction_metrics = run_predictions(
    manifest_state["val_rows"],
    config,
    tuned_bundle,
    name="tuned_val_predictions",
) if config.run_post_train_eval else None

test_prediction_metrics = run_predictions(
    manifest_state["test_rows"],
    config,
    tuned_bundle,
    name="tuned_test_predictions",
) if config.run_post_train_eval else None

print("Validation metrics:")
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print("Test metrics:")
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))

`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50358, 50359, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.


Validation metrics:
{
  "num_predictions": 766,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/eval_predictions/tuned_val_predictions.jsonl",
  "wer": 0.4996066782043998,
  "cer": 0.21857847544560263,
  "wer_loose": 0.476908012231551,
  "cer_loose": 0.21182447363608367,
  "wer_no_punct": 0.4708897394167643,
  "cer_no_punct": 0.19767091294230962,
  "total_hours": 1.0070916678090185,
  "by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "wer": 0.49978361987061426,
      "cer": 0.21931358232145162
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 9,
      "wer": 0.48472391805725135,
      "cer": 0.1567478193325261
    }
  },
  "object_dump_predictions": 0,
  "object_dump_prediction_uids": []
}
Test metrics:
{
  "num_predictions": 767,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/eval_predictions/tuned_test_predi

## 6. Summary Report

In [7]:
import json

from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics if "baseline_test_metrics" in globals() else None,
    val_prediction_metrics if "val_prediction_metrics" in globals() else None,
    test_prediction_metrics if "test_prediction_metrics" in globals() else None,
    training_summary,
)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))

{
  "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
  "notebook_path": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/whisper_medium_lora_levantine_custom_streaming_5minckpt_run.ipynb",
  "run_dir": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt",
  "selection_summary": {
    "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
    "drop_segments_at_or_above_seconds": 30.0,
    "full_counts_after_filter": {
      "train": 1541,
      "eval_parquet_pool": 1514,
      "eval_arrow_pool": 19
    },
    "selected_counts": {
      "train": 1541,
      "val": 766,
      "test": 767
    },
    "selected_hours": {
      "train": 2.1407003992305276,
      "val": 1.0070916678090185,
      "test": 1.037511566857676
    },
    "train_by_source_group": {
      "casablanca_levantine_train": {
        "rows": 1514,
        "hours": 2.0004557985360836
      },
      "omnilingual_apc_north_levantin

## 7. Integrity Report

In [ ]:
import json

from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report

integrity_report = write_integrity_report(
    config,
    selection_summary,
    baseline_test_metrics if "baseline_test_metrics" in globals() else None,
    val_prediction_metrics if "val_prediction_metrics" in globals() else None,
    test_prediction_metrics if "test_prediction_metrics" in globals() else None,
    training_summary,
)
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))

test_health = integrity_report.get("prediction_health", {}).get("test_predictions", {})
if test_health.get("object_dump_predictions", 0):
    raise RuntimeError("Decode guard failed: found object-dump predictions in test output.")
if test_health.get("empty_predictions", 0):
    raise RuntimeError("Integrity check failed: found empty predictions in test output.")

print("WHISPER MEDIUM MINI RUN INTEGRITY CHECK PASSED")

{
  "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
  "notebook_path": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt/whisper_medium_lora_levantine_custom_streaming_5minckpt_run.ipynb",
  "run_dir": "/home/MohammadNabulsi/whisper/Runs/whisper_medium_levantine_custom_streaming_5minckpt",
  "selection_summary": {
    "run_id": "whisper_medium_levantine_custom_streaming_5minckpt",
    "drop_segments_at_or_above_seconds": 30.0,
    "full_counts_after_filter": {
      "train": 1541,
      "eval_parquet_pool": 1514,
      "eval_arrow_pool": 19
    },
    "selected_counts": {
      "train": 1541,
      "val": 766,
      "test": 767
    },
    "selected_hours": {
      "train": 2.1407003992305276,
      "val": 1.0070916678090185,
      "test": 1.037511566857676
    },
    "train_by_source_group": {
      "casablanca_levantine_train": {
        "rows": 1514,
        "hours": 2.0004557985360836
      },
      "omnilingual_apc_north_levantin

: 